# Climate anxiety and flooding in Sylhet: analysis reproduction

This notebook reproduces every result reported in the manuscript from the raw survey file
`S1_Dataset.xlsx`. Running all cells top to bottom regenerates the reliability statistics,
the exploratory factor analysis, Tables 1 to 6, and Figures 1 to 6.

**How the pipeline is organized**

1. Load and recode the raw responses into numeric analysis variables.
2. Scale reliability (Cronbach's alpha) for the CCAS and its two subscales.
3. Exploratory factor analysis (KMO, Bartlett, variance explained, loadings).
4. Table 1 to Table 6.
5. Figures 1 to 6.

**Requirements**: `pandas`, `numpy`, `scipy`, `matplotlib`. The regression uses a small
hand-written HC3 robust standard-error routine so the results do not depend on `statsmodels`.

**A note on scoring choices.** The CCAS is scored as the mean of its items (range 1 to 5).
The three anxiety bands use equal thirds of that range: low below 2.33, moderate 2.33 to 3.67,
and high at 3.67 or above. In the hierarchical regression the continuous predictors and the
outcome are standardized (z-scored), while the two binary predictors (female, confides in
friends or neighbours) are kept as raw 0/1 dummies so their coefficients read as group
differences. The neighbourhood table (Table 6) reports the share scoring high using the
higher 3.5 cut, matching the manuscript.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 60)
np.random.seed(42)  # only affects the permutation test in Table 6

DATA_PATH = 'S1_Dataset.xlsx'   # adjust if the file sits elsewhere

## 1. Load the data

In [ ]:
raw = pd.read_excel(DATA_PATH, sheet_name='Data')
print('Respondents:', len(raw))
print('Columns     :', raw.shape[1])
raw.head(3)

## 2. Recode responses into analysis variables

Every categorical answer is mapped to the numeric coding used in the paper. The result is a
tidy frame `d` holding the CCAS scores, the anxiety band, and all regression predictors.

In [ ]:
# --- CCAS items -------------------------------------------------------------
likert = {'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4, 'Almost Always': 5}

cog_emo = ['CCAS_Q8_Concentrate','CCAS_Q9_Sleep','CCAS_Q10_Nightmares','CCAS_Q11_Crying',
           'CCAS_Q12_HandleBetter','CCAS_Q13_PreferAlone','CCAS_Q14_ReflectDeeply',
           'CCAS_Q15_QuestionReact']
functional = ['CCAS_Q16_FamilyFriends','CCAS_Q17_BalanceDaily','CCAS_Q18_WorkStudies',
              'CCAS_Q19_Potential','CCAS_Q20_OthersToldTooMuch']
ccas_items = cog_emo + functional

ccas_num = raw[ccas_items].replace(likert).apply(pd.to_numeric)

d = pd.DataFrame(index=raw.index)
d['ccas_total'] = ccas_num.mean(axis=1)
d['ccas_ce']    = ccas_num[cog_emo].mean(axis=1)
d['ccas_fn']    = ccas_num[functional].mean(axis=1)

# --- three-band anxiety category (equal thirds of the 1-5 range) -------------
LOW_CUT, HIGH_CUT = 1 + 4/3, 1 + 8/3   # 2.333, 3.667
def band(x):
    if x < LOW_CUT:  return 'Low'
    if x < HIGH_CUT: return 'Moderate'
    return 'High'
d['anxiety_band'] = d['ccas_total'].apply(band)

# --- predictors -------------------------------------------------------------
severity_map = {'Mild (Minor inconvenience)': 1,
                'Moderate (Damaged homes or roads, but not destructive)': 2,
                'Very Severe (Destroyed homes and infrastructure)': 3}
areatype_map = {'Elevated area': 1, 'Moderately flood-prone': 2, 'Floodplain/Low-lying area': 3}
freq_map     = {'Rarely': 1, 'Bi-annually': 2, 'Annually': 3}
coop_map     = {'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4, 'Always': 5}
agree_map    = {'Strongly Disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly Agree': 5}
edu_map      = {'No formal education': 1, 'Primary school': 2, 'Secondary/SSC/O-Level': 3,
                'Higher Secondary/HSC/A-Level': 4, "Bachelor's degree": 5,
                "Master's degree or higher": 6, 'Other': np.nan}

d['severity']  = raw['Q6_FloodSeverity'].map(severity_map)
d['area_type'] = raw['Q3_AreaType'].map(areatype_map)
d['frequency'] = raw['Q4_FloodFrequency'].map(freq_map)
d['coop']      = raw['Q23_CooperationFreq'].map(coop_map)
d['coop_reduces'] = raw['Q24_CooperationReducesImpact'].map(agree_map)
d['authorities']  = raw['Q28_AuthoritiesRespond'].map(agree_map)
d['education'] = raw['Q36_Education'].map(edu_map)
d['age']       = pd.to_numeric(raw['Age'], errors='coerce')
d['income']    = pd.to_numeric(raw['Q39_Income'], errors='coerce')

# number of loss types (exclude the free-text 'Others' bucket)
loss_cols = [c for c in raw.columns if c.startswith('Q7_Loss_') and c != 'Q7_Loss_Others']
d['n_losses'] = raw[loss_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)

# binary predictors kept as raw 0/1 dummies
d['female']  = (raw['Q40_Gender'] == 'Female').astype(float)
d['confide'] = (raw['Q25_TalkTo'] == 'Talk to friends or neighbors').astype(float)

d['neighbourhood'] = raw['Area']
d.head(3)

## 3. Scale reliability (Cronbach's alpha)

Reported in the manuscript as 0.88 for the full 13-item CCAS, 0.82 for the cognitive-emotional
subscale, and 0.85 for the functional subscale.

In [ ]:
def cronbach_alpha(frame):
    x = frame.dropna()
    k = x.shape[1]
    item_var = x.var(ddof=1, axis=0).sum()
    total_var = x.sum(axis=1).var(ddof=1)
    return k / (k - 1) * (1 - item_var / total_var)

print('Cronbach alpha')
print('  Full CCAS (13 items)        :', round(cronbach_alpha(ccas_num), 2))
print('  Cognitive-emotional (8)     :', round(cronbach_alpha(ccas_num[cog_emo]), 2))
print('  Functional (5)              :', round(cronbach_alpha(ccas_num[functional]), 2))

print('\nSubscale means (SD)')
for name, s in [('Total', d['ccas_total']), ('Cognitive-emotional', d['ccas_ce']),
                ('Functional', d['ccas_fn'])]:
    print(f'  {name:22s}: {s.mean():.2f} ({s.std(ddof=1):.2f})')

print('\nAnxiety bands (%)')
print((d['anxiety_band'].value_counts(normalize=True)[['Low','Moderate','High']]*100).round(1))

## 4. Exploratory factor analysis

KMO = 0.88, Bartlett's test chi-square(78) = 1482.7 (p < 0.001), two factors explaining
about 57.9% of the variance.

In [ ]:
R = ccas_num.dropna()
corr = np.corrcoef(R.values, rowvar=False)
p = corr.shape[0]
n = R.shape[0]

# Bartlett's test of sphericity
det = np.linalg.det(corr)
chi_sq = -(n - 1 - (2 * p + 5) / 6) * np.log(det)
df_bart = p * (p - 1) / 2
p_bart = stats.chi2.sf(chi_sq, df_bart)
print(f"Bartlett: chi-square({int(df_bart)}) = {chi_sq:.1f}, p = {p_bart:.3g}")

# Kaiser-Meyer-Olkin measure
inv = np.linalg.inv(corr)
d_ = np.sqrt(np.diag(inv))
partial = -inv / np.outer(d_, d_)          # anti-image / partial correlations
np.fill_diagonal(partial, 0)
off = ~np.eye(p, dtype=bool)
kmo = (corr[off]**2).sum() / ((corr[off]**2).sum() + (partial[off]**2).sum())
print(f"KMO overall = {kmo:.2f}")

# variance explained by the first two components
eigvals = np.sort(np.linalg.eigvalsh(corr))[::-1]
var_ratio = eigvals / p
print(f"Eigenvalues > 1: {(eigvals > 1).sum()}")
print(f"Variance explained by 2 factors: {var_ratio[:2].sum()*100:.1f}%")
print("Cumulative variance:", np.round(np.cumsum(var_ratio[:2]) * 100, 1))

### Two-factor loadings (varimax-style, principal components)

In [ ]:
# principal-component loadings for the first two factors, then a simple varimax rotation
vals, vecs = np.linalg.eigh(corr)
order = np.argsort(vals)[::-1][:2]
load = vecs[:, order] * np.sqrt(vals[order])

def varimax(L, iters=100, tol=1e-6):
    L = L.copy(); k = L.shape[1]; dsum = 0
    for _ in range(iters):
        u, s, vt = np.linalg.svd(
            L.T @ (L**3 - (L @ np.diag((L**2).sum(0)) / L.shape[0])))
        R = u @ vt; L = L @ R
        ns = s.sum()
        if abs(ns - dsum) < tol: break
        dsum = ns
    return L

Lr = varimax(load)
loadings = pd.DataFrame(Lr, index=ccas_items, columns=['Factor 1', 'Factor 2']).round(2)
print(loadings)

## 5. Table 1. Sample characteristics

In [ ]:
def cat_table(series, order=None):
    vc = series.value_counts(dropna=False)
    if order: vc = vc.reindex(order)
    pct = (vc / vc.sum() * 100).round(1)
    return pd.DataFrame({'n': vc.astype('Int64'), '%': pct})

print("Age: mean {:.1f}, SD {:.1f}, range {:.0f}-{:.0f}".format(
    d['age'].mean(), d['age'].std(ddof=1), d['age'].min(), d['age'].max()))
print("\nGender"); print(cat_table(raw['Q40_Gender']))
print("\nEducation")
print(cat_table(raw['Q36_Education'],
    ['No formal education','Primary school','Secondary/SSC/O-Level',
     'Higher Secondary/HSC/A-Level',"Bachelor's degree","Master's degree or higher",'Other']))
print("\nMarital status"); print(cat_table(raw['Q37_Marital']))
print("\nMonthly income (BDT): median {:.0f}, IQR {:.0f}-{:.0f}".format(
    d['income'].median(), d['income'].quantile(.25), d['income'].quantile(.75)))
print("\nFlood severity experienced")
print(cat_table(raw['Q6_FloodSeverity'],
    ['Mild (Minor inconvenience)',
     'Moderate (Damaged homes or roads, but not destructive)',
     'Very Severe (Destroyed homes and infrastructure)']))

## 6. Table 2. Bivariate associations with total climate anxiety

Spearman correlations for the continuous and ordinal predictors, with Benjamini-Hochberg
false-discovery-rate correction, plus one-way ANOVA for the categorical grouping variables.

In [ ]:
def bh_fdr(pvals):
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p); ranks = np.arange(1, m + 1)
    adj = np.empty(m)
    adj[order] = np.minimum.accumulate((p[order] * m / ranks)[::-1])[::-1]
    return np.clip(adj, 0, 1)

spearman_vars = ['severity','n_losses','area_type','frequency','coop','coop_reduces',
                 'authorities','age','income','education']
rows = []
for v in spearman_vars:
    pair = d[['ccas_total', v]].dropna()
    rho, pv = stats.spearmanr(pair['ccas_total'], pair[v])
    rows.append({'variable': v, 'n': len(pair), 'rho': round(rho, 2), 'p_raw': pv})
tab2 = pd.DataFrame(rows)
tab2['p_fdr'] = bh_fdr(tab2['p_raw'].values).round(3)
tab2['p_raw'] = tab2['p_raw'].round(3)
print(tab2.to_string(index=False))

# ANOVA for categorical group comparisons
print("\nOne-way ANOVA (total anxiety by group)")
for grp in ['Q40_Gender','Q25_TalkTo','Q6_FloodSeverity']:
    g = pd.DataFrame({'y': d['ccas_total'], 'g': raw[grp]}).dropna()
    groups = [x['y'].values for _, x in g.groupby('g')]
    F, pv = stats.f_oneway(*groups)
    print(f"  {grp:18s}: F = {F:.2f}, p = {pv:.3g}")

## 7. Table 3. Hierarchical regression on standardized climate anxiety

Three blocks entered in sequence. Continuous predictors and the outcome are z-scored; the
binary predictors (female, confide) stay as 0/1 dummies. Standard errors are HC3
heteroskedasticity-robust. Reported model fit: Block 1 R2 = 0.02, Block 2 R2 = 0.21,
Block 3 R2 = 0.30 (adjusted 0.26).

In [ ]:
def zscore(s):
    s = pd.to_numeric(s, errors='coerce')
    return (s - s.mean()) / s.std(ddof=1)

# build the design matrix pieces
Z = pd.DataFrame(index=d.index)
for v in ['age','income','education','severity','n_losses','area_type','frequency',
          'coop','authorities','coop_reduces']:
    Z[v] = zscore(d[v])
Z['female']  = d['female']    # dummy, unscaled
Z['confide'] = d['confide']   # dummy, unscaled
y = zscore(d['ccas_total'])

block1 = ['age','income','education','female']
block2 = block1 + ['severity','n_losses','area_type','frequency']
block3 = block2 + ['confide','coop','authorities','coop_reduces']

def ols_hc3(y, Xcols, data):
    dat = pd.concat([y.rename('y'), data[Xcols]], axis=1).dropna()
    yv = dat['y'].values
    X = np.column_stack([np.ones(len(dat)), dat[Xcols].values])
    XtXi = np.linalg.inv(X.T @ X)
    beta = XtXi @ X.T @ yv
    resid = yv - X @ beta
    h = np.einsum('ij,jk,ik->i', X, XtXi, X)      # leverage
    S = X * (resid / (1 - h))[:, None]
    cov = XtXi @ (S.T @ S) @ XtXi                 # HC3 sandwich
    se = np.sqrt(np.diag(cov))
    tval = beta / se
    dfree = len(dat) - X.shape[1]
    pval = 2 * stats.t.sf(np.abs(tval), dfree)
    ss_res = (resid**2).sum(); ss_tot = ((yv - yv.mean())**2).sum()
    r2 = 1 - ss_res / ss_tot
    adj = 1 - (1 - r2) * (len(dat) - 1) / dfree
    out = pd.DataFrame({'beta': beta[1:].round(3), 'SE': se[1:].round(3),
                        'p': pval[1:].round(3)}, index=Xcols)
    return out, r2, adj, len(dat)

prev = 0
for i, blk in enumerate([block1, block2, block3], 1):
    res, r2, adj, N = ols_hc3(y, blk, Z)
    print(f"===== Block {i}  (N = {N}) =====")
    print(res.to_string())
    print(f"R2 = {r2:.3f}   adjusted R2 = {adj:.3f}   delta R2 = {r2 - prev:.3f}\n")
    prev = r2

## 8. Table 4. Subscale-specific regressions

The full Block 3 model refit separately on the cognitive-emotional and functional subscale
scores (each standardized).

In [ ]:
for label, col in [('Cognitive-emotional', 'ccas_ce'), ('Functional', 'ccas_fn')]:
    ysub = zscore(d[col])
    res, r2, adj, N = ols_hc3(ysub, block3, Z)
    print(f"===== {label} subscale  (N = {N}) =====")
    print(res.to_string())
    print(f"R2 = {r2:.3f}   adjusted R2 = {adj:.3f}\n")

## 9. Table 5. Ranked community priorities

Respondents ranked four priorities from 1 (most important) to 4. The labels in the raw file
have a truncated first character, which is repaired here. Priorities are scored with a
weighted (Borda) count: rank 1 = 4 points down to rank 4 = 1 point.

In [ ]:
label_fix = {
    'ocial support from family and friends': 'Social support from family and friends',
    'overnment disaster management': 'Government disaster management',
    'Strong community cooperation': 'Strong community cooperation',
    'Better urban planning and infrastructure': 'Better urban planning and infrastructure',
}
rank_cols = ['Q33_Rank1','Q33_Rank2','Q33_Rank3','Q33_Rank4']
weights = {'Q33_Rank1': 4, 'Q33_Rank2': 3, 'Q33_Rank3': 2, 'Q33_Rank4': 1}

scores, first_choice = {}, {}
for c in rank_cols:
    for item, cnt in raw[c].map(label_fix).value_counts().items():
        scores[item] = scores.get(item, 0) + cnt * weights[c]
first_choice = raw['Q33_Rank1'].map(label_fix).value_counts()

tab5 = pd.DataFrame({'weighted_score': pd.Series(scores),
                     'ranked_first_n': first_choice}).sort_values('weighted_score', ascending=False)
tab5['mean_rank_points'] = (tab5['weighted_score'] / raw[rank_cols].notna().all(axis=1).sum()).round(2)
print(tab5.to_string())

## 10. Table 6. Neighbourhood-level anxiety and flood exposure

Mean anxiety and the share scoring high (using the 3.5 cut, as in the manuscript) for each of
the 13 neighbourhoods, the intraclass correlation, and a permutation-based one-way ANOVA.

In [ ]:
nb = pd.DataFrame({'area': d['neighbourhood'], 'ccas': d['ccas_total'],
                   'severity': d['severity']}).dropna(subset=['area','ccas'])

HIGH_TABLE6 = 3.5
grp = nb.groupby('area')['ccas']
tab6 = pd.DataFrame({
    'n': grp.size(),
    'mean_anxiety': grp.mean().round(2),
    'sd': grp.std(ddof=1).round(2),
    'pct_high': (nb.assign(hi=nb['ccas'] >= HIGH_TABLE6)
                   .groupby('area')['hi'].mean() * 100).round(1),
}).sort_values('mean_anxiety', ascending=False)
print(tab6.to_string())

# one-way random-effects ICC
groups = [g.values for _, g in grp]
k = len(groups); N = sum(len(g) for g in groups); grand = nb['ccas'].mean()
ss_between = sum(len(g) * (g.mean() - grand)**2 for g in groups)
ss_within  = sum(((g - g.mean())**2).sum() for g in groups)
ms_between = ss_between / (k - 1)
ms_within  = ss_within / (N - k)
n0 = (N - sum(len(g)**2 for g in groups) / N) / (k - 1)
icc = (ms_between - ms_within) / (ms_between + (n0 - 1) * ms_within)
F_obs = ms_between / ms_within
print(f"\nICC (one-way) = {icc:.2f}")
print(f"ANOVA F({k-1},{N-k}) = {F_obs:.2f}")

# permutation p-value for the neighbourhood effect
labels = nb['area'].values; vals = nb['ccas'].values
perm = np.empty(5000)
for i in range(5000):
    pv = np.random.permutation(vals)
    gm = pd.Series(pv).groupby(labels).mean()
    gs = pd.Series(pv).groupby(labels).size()
    ssb = (gs * (gm - pv.mean())**2).sum()
    ssw = sum(((pv[labels == a] - pv[labels == a].mean())**2).sum() for a in gm.index)
    perm[i] = (ssb / (k - 1)) / (ssw / (N - k))
print(f"Permutation p = {(perm >= F_obs).mean():.4f}")

## 11. Figures

The colours use the Okabe-Ito colour-blind-safe palette used in the manuscript.

In [ ]:
OKABE = ['#0072B2','#E69F00','#009E73','#D55E00','#CC79A7','#56B4E9','#F0E442','#000000']
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.bbox': 'tight'})

### Figure 1. Reported loss types

In [ ]:
loss_counts = (raw[[c for c in raw.columns if c.startswith('Q7_Loss_') and c != 'Q7_Loss_Others']]
               .apply(pd.to_numeric, errors='coerce').sum().sort_values())
labels = [c.replace('Q7_Loss_', '') for c in loss_counts.index]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(labels, loss_counts.values, color=OKABE[0])
ax.set_xlabel('Number of respondents reporting the loss')
ax.set_title('Figure 1. Flood-related losses reported')
plt.show()

### Figure 2. Distribution of total climate anxiety

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(d['ccas_total'].dropna(), bins=np.arange(1, 5.2, 0.25),
        color=OKABE[5], edgecolor='white')
for c, lab in [(LOW_CUT, 'low/moderate'), (HIGH_CUT, 'moderate/high')]:
    ax.axvline(c, color=OKABE[3], ls='--', lw=1)
    ax.text(c, ax.get_ylim()[1]*0.9, f' {lab}', color=OKABE[3], fontsize=8)
ax.set_xlabel('Mean CCAS score (1-5)'); ax.set_ylabel('Respondents')
ax.set_title('Figure 2. Distribution of climate anxiety scores')
plt.show()

### Figure 3. Anxiety by flood severity

In [ ]:
order = ['Mild (Minor inconvenience)',
         'Moderate (Damaged homes or roads, but not destructive)',
         'Very Severe (Destroyed homes and infrastructure)']
short = ['Mild', 'Moderate', 'Very severe']
data = [d.loc[raw['Q6_FloodSeverity'] == s, 'ccas_total'].dropna().values for s in order]
fig, ax = plt.subplots(figsize=(6.5, 4.5))
bp = ax.boxplot(data, labels=short, patch_artist=True, showmeans=True)
for patch, c in zip(bp['boxes'], OKABE): patch.set_facecolor(c); patch.set_alpha(.6)
ax.set_ylabel('Mean CCAS score'); ax.set_xlabel('Flood severity experienced')
ax.set_title('Figure 3. Climate anxiety by flood severity')
plt.show()

### Figure 4. Standardized regression coefficients (Block 3 forest plot)

In [ ]:
res3, _, _, _ = ols_hc3(y, block3, Z)
res3 = res3.iloc[::-1]
ci = 1.96 * res3['SE']
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.errorbar(res3['beta'], range(len(res3)), xerr=ci, fmt='o',
            color=OKABE[0], ecolor=OKABE[0], capsize=3)
ax.axvline(0, color='grey', lw=1)
ax.set_yticks(range(len(res3))); ax.set_yticklabels(res3.index)
ax.set_xlabel('Standardized beta (95% CI, HC3)')
ax.set_title('Figure 4. Predictors of climate anxiety')
plt.show()

### Figure 5. Community priorities

In [ ]:
tab5_plot = tab5.sort_values('weighted_score')
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.barh([t.replace(' and ', ' &\n') for t in tab5_plot.index],
        tab5_plot['weighted_score'], color=OKABE[2])
ax.set_xlabel('Weighted priority score (rank 1 = 4 ... rank 4 = 1)')
ax.set_title('Figure 5. Ranked community priorities')
plt.show()

### Figure 6. Neighbourhood anxiety and flood exposure

In [ ]:
sev_by_area = nb.groupby('area')['severity'].mean()
plot6 = tab6.join(sev_by_area.rename('mean_severity'))
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(plot6['mean_severity'], plot6['mean_anxiety'],
                s=plot6['n'] * 6, c=plot6['pct_high'], cmap='YlOrRd',
                edgecolor='black', linewidth=.5)
for a, r in plot6.iterrows():
    ax.annotate(a, (r['mean_severity'], r['mean_anxiety']),
                fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Mean flood severity'); ax.set_ylabel('Mean climate anxiety')
ax.set_title('Figure 6. Neighbourhood anxiety vs flood exposure')
fig.colorbar(sc, label='% scoring high')
plt.show()